In [1]:
from pathlib import Path 
import pandas as pd 

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pm = pd.read_csv(RAIZ / "data" / "processed" / "pm25_diario_2017_2026.csv",
                 index_col="fecha", parse_dates=True)

largo = pm.stack().reset_index()
largo.columns = ["fecha", "estacion", "pm25"]
largo.to_csv(RAIZ / "data" / "processed" / "pm25_largo.csv", index=False)

print(largo.shape)
largo.head()

(33760, 3)


,fecha,estacion,pm25
0,2017-01-01,carvajal,13.460870
1,2017-01-01,guaymaral,7.666667
2,2017-01-01,kennedy,NaN
3,2017-01-01,lasferias,NaN
4,2017-01-01,minambiente,4.600000


In [3]:
print(largo["pm25"].isna().sum())

largo = largo.dropna(subset=["pm25"])
largo.to_csv(RAIZ / "data" / "processed" / "pm25_largo.csv", index=False)
print(largo.shape)

1683
(32077, 3)


In [4]:
import duckdb

ruta = RAIZ / "data" / "processed" / "pm25_largo.csv"

duckdb.sql(f"""
    SELECT estacion,
           COUNT(*)            AS dias,
           ROUND(AVG(pm25), 1) AS promedio
    FROM read_csv_auto('{ruta}')
    GROUP BY estacion
    ORDER BY promedio DESC
""").df()

,estacion,dias,promedio
0,carvajal,2761,31.3
1,kennedy,3288,21.7
2,tunal,3295,17.2
3,paranda,3206,16.2
4,suba,3307,15.2
5,lasferias,3300,14.4
6,guaymaral,3295,13.6
7,minambiente,3232,13.5
8,scristobal,3244,12.1
9,usaquen,3149,11.8


In [5]:
duckdb.sql("""
    WITH dias_completos AS (
        SELECT fecha
        FROM largo
        GROUP BY fecha
        HAVING COUNT(*) = 10
    )
    SELECT estacion,
           COUNT(*)            AS dias,
           ROUND(AVG(pm25), 1) AS promedio
    FROM largo
    JOIN dias_completos USING (fecha)
    GROUP BY estacion
    ORDER BY promedio DESC
""").df()

,estacion,dias,promedio
0,carvajal,2005,31.6
1,kennedy,2005,21.5
2,tunal,2005,16.7
3,paranda,2005,16.3
4,suba,2005,15.1
5,lasferias,2005,14.2
6,minambiente,2005,13.5
7,guaymaral,2005,13.3
8,usaquen,2005,11.7
9,scristobal,2005,11.4


In [6]:
duckdb.sql("""
    WITH completos AS (
        SELECT fecha FROM largo
        GROUP BY fecha
        HAVING COUNT(*) = 10
    ),
    todos AS (
        SELECT estacion, AVG(pm25) AS p
        FROM largo
        GROUP BY estacion
    ),
    justos AS (
        SELECT estacion, AVG(pm25) AS p
        FROM largo JOIN completos USING (fecha)
        GROUP BY estacion
    )
    SELECT estacion,
           ROUND(todos.p, 1)            AS todos_los_dias,
           ROUND(justos.p, 1)           AS dias_completos,
           ROUND(justos.p - todos.p, 1) AS diferencia
    FROM todos JOIN justos USING (estacion)
    ORDER BY dias_completos DESC
""").df()

,estacion,todos_los_dias,dias_completos,diferencia
0,carvajal,31.3,31.6,0.3
1,kennedy,21.7,21.5,-0.1
2,tunal,17.2,16.7,-0.4
3,paranda,16.2,16.3,0.1
4,suba,15.2,15.1,-0.1
5,lasferias,14.4,14.2,-0.2
6,minambiente,13.5,13.5,0.0
7,guaymaral,13.6,13.3,-0.3
8,usaquen,11.8,11.7,-0.1
9,scristobal,12.1,11.4,-0.6
